# 04 — Conversion Analysis

This notebook measures purchase conversion and compares rates across the segments available in the Retailrocket-derived dataset.

## Coverage
- Conversion outcome definition
- Cohort and observation windows
- Segment conversion rates
- Confidence intervals
- Lift and statistical comparisons
- Export of conversion analysis tables

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "processed" / "ecommerce_funnel_events_clean.csv"
if not DATA_PATH.exists():
    DATA_PATH = ROOT / "data" / "ecommerce_funnel_events.csv"
REPORTS_PATH = ROOT / "reports"
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
df["event_timestamp"] = pd.to_datetime(df["event_timestamp"], utc=True, errors="coerce")

all_users = set(df["user_id"])
product_users = set(df.loc[df["stage"] == "product_view", "user_id"])
cart_users = set(df.loc[df["stage"] == "cart", "user_id"])
purchase_users = set(df.loc[df["stage"] == "purchase", "user_id"])

overall_rate = len(purchase_users & product_users) / len(product_users) if product_users else 0.0
cart_rate = len(purchase_users & cart_users) / len(cart_users) if cart_users else 0.0
view_to_cart = len(cart_users & product_users) / len(product_users) if product_users else 0.0

print({
    "product_view_users": len(product_users),
    "cart_users": len(cart_users),
    "purchasers": len(purchase_users),
    "view_to_purchase_rate": overall_rate,
    "view_to_cart_rate": view_to_cart,
    "cart_to_purchase_rate": cart_rate,
})

segment_results = []
for column in ["campaign", "channel", "device"]:
    if column not in df.columns:
        continue
    for segment, segment_frame in df.groupby(column):
        segment_users = set(segment_frame["user_id"])
        views = segment_users & product_users
        purchasers = segment_users & purchase_users
        rate = len(purchasers) / len(views) if views else 0.0
        segment_results.append({"dimension": column, "segment": str(segment), "view_users": len(views), "purchasers": len(purchasers), "conversion_rate": rate})

segment_metrics = pd.DataFrame(segment_results)
if not segment_metrics.empty:
    segment_metrics["lift_vs_overall"] = segment_metrics["conversion_rate"] - overall_rate
    display(segment_metrics.sort_values(["dimension", "conversion_rate"], ascending=[True, False]))

    fig, ax = plt.subplots(figsize=(10, 5))
    plot_frame = segment_metrics[segment_metrics["segment"].astype(str) != "unknown"]
    if plot_frame.empty:
        plot_frame = segment_metrics
    plot_frame.plot(kind="bar", x="segment", y="conversion_rate", ax=ax, legend=False)
    ax.set_title("Observed Conversion Rate by Segment")
    ax.set_ylabel("Product-view to purchase rate")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    segment_metrics.to_csv(REPORTS_PATH / "conversion_segment_metrics.csv", index=False)

successes = len(purchase_users & product_users)
trials = len(product_users)
if trials:
    z = 1.96
    p = successes / trials
    denominator = 1 + z**2 / trials
    center = (p + z**2 / (2 * trials)) / denominator
    margin = z * np.sqrt(p * (1 - p) / trials + z**2 / (4 * trials**2)) / denominator
    print("Wilson 95% confidence interval:", round(max(0, center - margin), 4), round(min(1, center + margin), 4))